# 01 — The data

---

### The question

Ye et al. 2026 (*Nature Microbiology*) measured GC content and GC-containing dinucleotide frequencies across 115,520 influenza A genomes, and found both **declined over time** within lineages that had achieved sustained transmission in mammals.

Applying the same measurements to human RSV:

> **Within each RSV lineage, how does genome composition change over time, and does the pattern differ between RSV-A and RSV-B?**

This notebook covers the dataset: size, coverage in time and place, and what QC removes.

---

In [ ]:
import sys, os, subprocess
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = os.path.abspath("..")          # repo root; the R scripts expect to run from here
pd.set_option("display.width", 130)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
PAL = {"A": "#2C7FB8", "B": "#C62824"}    # same colours as the R figures

def run_r(script="R/figures.R"):
    """Run the R figure script from the repo root and report what it wrote."""
    r = subprocess.run(["Rscript", script], capture_output=True, text=True, cwd=ROOT)
    print(r.stdout.strip() or r.stderr.strip()[-400:])

def show(fig):
    """Display a figure from results/figures/ inline."""
    display(Image(os.path.join(ROOT, "results", "figures", fig)))


## 1.1  The sequences

`rsv_features.tsv` is written by `scripts/build_features.py`: one row per genome, GISAID metadata alongside composition features.

In [ ]:
d = pd.read_csv("../data/processed/rsv_features.tsv", sep="\t")
print("genomes :", len(d))
print("subtypes:", d.Subtype.value_counts().to_dict())
print("years   :", d.year.min(), "to", d.year.max())
# accession IDs are dropped from displays so no GISAID record is stored here
d.drop(columns=["Accession ID"]).head(3)

## 1.2  Quality control

An RSV genome is about 15,200 nucleotides. We keep a sequence only if it is

| Check | Threshold | Why |
|---|---|---|
| Length | 14,900–15,400 nt | keeps complete genomes, drops fragments |
| Ambiguous bases | at most 1% | `N` positions would distort letter counts |
| Collection date | complete | needed to place the genome in time |

The filter is deliberately gentle, because the GISAID download was already restricted to complete, high-coverage genomes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for s in "AB":
    ax.hist(d.loc[d.Subtype == s, "seq_len_actual"], bins=110,
            alpha=.6, label=f"RSV-{s}", color=PAL[s])
for x in (14900, 15400):
    ax.axvline(x, ls="--", lw=1, c="k")
ax.set_xlim(14700, 15600)
ax.set_xlabel("genome length (nt)"); ax.set_ylabel("genomes")
ax.set_title("Genome length, dashed lines mark the filter window")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

**Two genomes out of 33,040 were removed.** One RSV-B had 2.11% ambiguous bases. The other was an RSV-B at 15,854 nt — 632 nt over the reference, carrying a 583-nt non-RSV tail past the 3' end at 49% GC against the virus's 34%, i.e. contamination. That second case is why the length check has an upper bound and not only a minimum.

## 1.3  When and where the sequences come from

Two sampling features matter for everything downstream:

- **Not even over time** — rises steeply from 2018, dips in 2020–21 when COVID-19 disrupted both RSV circulation and surveillance.
- **Not even over the world** — Europe contributes far more genomes than anywhere else.

So any trend over calendar time is partly a trend in *who was sequencing*. This is the main reason notebook 03 fits inside lineages rather than across the whole dataset.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
yr = d.groupby(["year", "Subtype"]).size().unstack(fill_value=0)
yr.plot.bar(stacked=True, ax=ax[0], color=[PAL["A"], PAL["B"]], width=.85,
            legend=False)
ax[0].set_title("genomes per year"); ax[0].set_xlabel("")
ax[0].axvspan(yr.index.get_loc(2020) - .5, yr.index.get_loc(2021) + .5,
              color="grey", alpha=.18)
reg = d.groupby(["region", "Subtype"]).size().unstack(fill_value=0).sort_values("A")
reg.plot.barh(stacked=True, ax=ax[1], color=[PAL["A"], PAL["B"]])
ax[1].set_title("genomes per region"); ax[1].set_ylabel("")
ax[1].legend(title="subtype", frameon=False)
plt.tight_layout(); plt.show()

*Grey band marks the COVID-19 disruption.*

## 1.4  Known problems with this dataset

Four, found by auditing before analysing. Two change what we do.

| | Problem | Decision |
|---|---|---|
| **a** | 2026 tail incomplete (deposition lag) | cut the window at end of May 2026 |
| **b** | Sequencing geography shifted, even inside lineages | region as a covariate |
| **c** | Strong seasonality | noted, not corrected |
| **d** | Sparse early data gives single genomes high leverage | trim sparse lineage ends |

---

### (a) The 2026 tail is incomplete

The download was made in October 2026, but sequences take months to be deposited. Comparing 2026 month by month against the same months of 2025 separates deposition lag from ordinary seasonality.

In [ ]:
m = {y: d[d.date.str.startswith(str(y))].date.str.slice(5, 7).value_counts().sort_index()
     for y in (2025, 2026)}
cmp = pd.DataFrame(m).fillna(0).astype(int)
cmp["2026 as % of 2025"] = (cmp[2026] / cmp[2025] * 100).round(0)
print(cmp.to_string())
print("\nlast collection date:", d.date.max())

- Through May, 2026 holds **78–93%** of the previous year’s count — normal, and May itself is at **92%**.
- June falls to **53%**, July **29%**, August **4%** — eighteen genomes where 2025 had 399 in the same month.
- That is deposition lag, not season: those months are a non-random subset contributed only by laboratories that deposit quickly.

**Decision: the window runs 2001 to the end of May 2026.** May is the last month still at full strength, so it is the natural cut. Everything earlier is kept — sequencing effort grows steeply over the period, but uneven sample sizes across time do not bias a slope, they only make later years more precisely estimated.

---

### (b) Where sequencing happens has shifted completely

The more serious problem.

In [ ]:
reg = pd.crosstab(d.year, d.region, normalize="index").mul(100).round(1)
reg.loc[2015:2026]

- Oceania supplied **38%** of genomes in 2015 and **4%** in 2026; Europe went from **18%** to **56%**. North America contributes nothing at all in 2017.
- So calendar time and geography are **entangled** — a composition trend over time could partly be a trend in which part of the world was sequencing.
- The drift happens *inside* individual lineages too, so fitting within lineages does not remove it. Lineage A.D goes from 71% Oceania early to 36% Europe late.

But a shifting region mix only matters if composition actually differs by region. So: the composition across all years, and the same measure split by region.

In [ ]:
w = d[d.year.between(2015, 2026)]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))

# left: every genome pooled by year, ignoring lineage structure entirely
for s in "AB":
    t = w[w.Subtype == s].groupby("year").oe_CG.mean()
    ax[0].plot(t.index, t.values, "-o", ms=4, color=PAL[s], label=f"RSV-{s}")
ax[0].set_xlabel("year"); ax[0].set_ylabel("mean CpG O/E")
ax[0].set_title("Composition across all years, pooled")
ax[0].legend(frameon=False)

# right: the same measure by region, to see whether region carries any signal
r = w.pivot_table(index="region", columns="Subtype", values="oe_CG", aggfunc="mean")
r.plot.barh(ax=ax[1], color=[PAL["A"], PAL["B"]])
ax[1].legend(["RSV-A", "RSV-B"], frameon=False, loc="lower right")
ax[1].set_xlim(0.230, 0.245); ax[1].set_ylabel("")
ax[1].set_xlabel("mean CpG O/E"); ax[1].set_title("The same measure, by region")
plt.tight_layout(); plt.show()

print("spread across regions (max - min), within subtype:")
for v in ["gc_content", "oe_CG"]:
    for s in "AB":
        x = w[w.Subtype == s].groupby("region")[v].mean()
        print(f"  {v:11s} RSV-{s}: {x.max() - x.min():.5f}")

Both halves matter:

- **The pooled line does move** — RSV-B's CpG O/E climbs from 0.2357 in 2015 to 0.2430 in 2023. Taken at face value that is a trend, and it is exactly the kind of number that section 3.1 explains you cannot trust, because lineages are replacing one another underneath it.
- **Region carries real signal.** Within RSV-B the regional spread in CpG O/E is **0.0070**; the largest within-lineage slope anywhere in this project is 0.0011 per year. So the gap between two regions is worth roughly six years of trend. A shift in which regions were sequencing can therefore imitate a time trend.

**Decision: region enters the regressions as a covariate, and results are reported both ways** (notebook 03).

---

### (c) Strong seasonality — noted, not corrected

RSV is seasonal, so genomes are not spread evenly through the year.

In [ ]:
mo = d.date.str.slice(5, 7).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.bar(range(1, 13), mo.values, color="#6B8FA8", edgecolor="grey", linewidth=.3)
ax.set_xticks(range(1, 13)); ax.set_xlabel("month"); ax.set_ylabel("genomes")
ax.set_title("Collection month, all years pooled")
plt.tight_layout(); plt.show()

November and December carry three to four times September. The two hemispheres also peak at opposite times, so season and geography are partly entangled as well. Minor for a slope measured across a decade, so we record it rather than correct it.

---

### (d) Sparse early data gives single genomes high leverage

- Everything before 2015 is about 3% of the dataset, and **RSV-A has essentially no sequences before 2010**.
- Keeping it is still worth doing: for some lineages it adds years of span, and a longer span makes a slope more precise.
- But the leverage is dangerous — lineage A.D.3 would reach back **6.9 extra years on the strength of one genome from 2010**, and a point that isolated drags the fitted line hard.

**Decision: each lineage is cut back to the run of years in which it was actually being sampled**, dropping leading and trailing years holding fewer than 5 genomes. The rule, and what it does to each lineage:

In [ ]:
f = pd.read_csv("../data/processed/rsv_features.tsv", sep="\t")
f["lineage_grp"] = f.Lineage.astype(str).str.split(".").str[:3].str.join(".")
f["yr"] = f.date.str.slice(0, 4).astype(int)

rows = []
for (sub, grp), s in f.groupby(["Subtype", "lineage_grp"]):
    counts = s.yr.value_counts().sort_index()
    ok = counts[counts >= 5].index                      # years holding >= 5 genomes
    if len(ok) == 0:
        continue
    kept = s[(s.yr >= ok.min()) & (s.yr <= ok.max())]   # keep the run between them
    if kept.yr.nunique() >= 5 and len(kept) >= 200:     # the inclusion rule
        rows.append({"Subtype": sub, "lineage": grp,
                     "first yr raw": s.yr.min(), "first yr kept": kept.yr.min(),
                     "yrs cut": kept.yr.min() - s.yr.min(),
                     "genomes cut": len(s) - len(kept)})
print(pd.DataFrame(rows).sort_values(["Subtype", "lineage"]).to_string(index=False))

# well-sampled early data is not touched by the rule
ad = f[(f.Subtype == "A") & (f.lineage_grp == "A.D")]
c = ad.yr.value_counts().sort_index(); ok = c[c >= 5].index
kept = ad[(ad.yr >= ok.min()) & (ad.yr <= ok.max())]
dt = pd.to_datetime(kept.date)
print("\nA.D genomes within 3 years of its first:",
      int((dt < dt.min() + pd.DateOffset(years=3)).sum()))

**A.D.3 loses 7 years and only 1 genome** — that is the whole point of the rule: it removes span, not data. A.D keeps **361 genomes in its first three years**, so well-sampled early data is untouched.

This is the same rule `lineage.prepare()` applies in notebooks 03 and 04; the kept start years here match `results/lineages.tsv` exactly. It lives in `src/rsv/lineage.py` only because those notebooks and the analysis script all need it.

## 1.5  Lineages

GISAID assigns each genome to a **lineage**, a named branch of the RSV family tree. Labels are hierarchical and dot-separated: `A.D.1.5` sits inside `A.D.1`, which sits inside `A.D`.

Lineages rise and fall over a few seasons. Notebook 03 covers why that matters and how the level is chosen.

In [ ]:
for s in "AB":
    top = d[d.Subtype == s].Lineage.value_counts().head(5)
    print(f"RSV-{s} most common lineage labels")
    print(top.to_string(), "\n")